# proteomics_analysis

## Objective
Run the standardized proteomics workflow from the template configuration files.

This notebook is the main notebook counterpart to the shared script set in `workflow/scripts/python/`. It reads project-specific settings from `workflow/00_raw_data/config/`, previews the main processing steps in-notebook, and writes outputs into the corresponding workflow step folders.


## Inputs
- `workflow/00_raw_data/config/project_manifest.yaml`
- `workflow/00_raw_data/config/sample_metadata.csv`
- `workflow/00_raw_data/config/comparisons.csv`

## Main outputs
- `workflow/01_qc_normalization/`
- `workflow/02_statistics/`
- `workflow/03_visualization/`


# Import libraries


In [ ]:
import os
import re
import sys
import json
import shutil
from pathlib import Path
from functools import reduce

import yaml
import numpy as np
import pandas as pd
import plotly
import scipy
import sklearn
import statsmodels
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns


# Add python scripts to import path


In [ ]:
sys.path.append('../python')


# Import current shared functions


In [ ]:
from helpers import extractAnnotation, renameColumns, get_columns_for_group
from filter_and_Normalize import (
    filterLowPSM,
    extractAbundanceData,
    perform_prtc_pca,
    normalize_prtc,
    uq_normalization,
    plot_data_distribution,
)
from Stats import perform_student_t_test, calculate_foldchange, test_shapiro_wilk_normality
from volcano_plotter import plot_volcano
from heatmap_plotter import plot_heatmap
from pca_plotter import compute_pca, plot_all_2d_pca_pairs, plot_pca_3d, plot_pls_da_with_cv


# Load project configuration


In [ ]:
notebook_dir = Path.cwd().resolve()
workflow_root = notebook_dir.parent.parent
project_root = workflow_root.parent

manifest_path = workflow_root / '00_raw_data' / 'config' / 'project_manifest.yaml'
sample_metadata_path = workflow_root / '00_raw_data' / 'config' / 'sample_metadata.csv'
comparisons_path = workflow_root / '00_raw_data' / 'config' / 'comparisons.csv'

with open(manifest_path, 'r', encoding='utf-8') as handle:
    manifest = yaml.safe_load(handle)

sample_metadata = pd.read_csv(sample_metadata_path)
comparisons = pd.read_csv(comparisons_path)

project_cfg = manifest['project']
input_cfg = manifest['inputs']
analysis_cfg = manifest['analysis']
output_cfg = manifest['outputs']
flags_cfg = manifest['flags']

provider_export = project_root / input_cfg['provider_export']
if input_cfg.get('cleaned_export') and str(input_cfg['cleaned_export']).strip():
    cleaned_export = project_root / input_cfg['cleaned_export']
else:
    cleaned_export = None

data_path = cleaned_export if cleaned_export else provider_export
qc_dir = project_root / output_cfg['qc_dir']
results_dir = project_root / output_cfg['results_dir']
plots_dir = project_root / output_cfg['plots_dir']
secondary_dir = project_root / output_cfg['secondary_dir']

results_csv_dir = results_dir / 'CSV'
plots_output_dir = plots_dir / 'output'
plots_png_dir = plots_output_dir / 'PNG'
plots_svg_dir = plots_output_dir / 'SVG'
plots_html_dir = plots_output_dir / 'HTML'

for path in [qc_dir, results_dir, plots_dir, secondary_dir, results_csv_dir, plots_output_dir, plots_png_dir, plots_svg_dir, plots_html_dir]:
    path.mkdir(parents=True, exist_ok=True)

sample_metadata


# Validate project configuration


In [ ]:
required_sample_metadata = {'sample_id', 'source_column', 'treatment', 'group', 'include'}
required_comparison_columns = {'comparison_id', 'grouping_column', 'group1', 'group2', 'enabled', 'pvalue_cutoff', 'qvalue_cutoff', 'abs_log2fc_cutoff'}

missing_sm = required_sample_metadata.difference(sample_metadata.columns)
missing_cmp = required_comparison_columns.difference(comparisons.columns)

if missing_sm:
    raise KeyError(f'Missing required sample metadata columns: {sorted(missing_sm)}')
if missing_cmp:
    raise KeyError(f'Missing required comparison columns: {sorted(missing_cmp)}')
if not data_path.exists():
    raise FileNotFoundError(f'Input data file not found: {data_path}')

print('Project:', project_cfg['project_id'])
print('Title:', project_cfg['project_title'])
print('Input data:', data_path)
print('QC dir:', qc_dir)
print('Results dir:', results_dir)
print('Plots dir:', plots_dir)
print('Enabled comparisons:', int(comparisons['enabled'].astype(str).str.lower().isin(['true', '1', 'yes', 'y']).sum()))


# Read data


In [ ]:
input_format = str(analysis_cfg.get('input_format', 'xlsx')).lower()
index_col = analysis_cfg.get('input_index_column', 4)

if input_format in {'xlsx', 'xls'}:
    data = pd.read_excel(data_path, index_col=index_col)
elif input_format == 'csv':
    data = pd.read_csv(data_path, index_col=index_col)
else:
    raise ValueError(f'Unsupported input_format: {input_format}')

data.shape


# Extract annotation columns


In [ ]:
annotation = extractAnnotation(data)
annotation.head()


# Optional project-specific row filtering


In [ ]:
# Apply any project-specific row filtering here.
# Example patterns:
# - retain one organism while keeping PRTC rows
# - retain one tissue subset
# - drop known contaminant rows
#
# Example:
# keep_mask = data['Description'].str.contains('Streptococcus', case=False, na=False)
# prtc_mask = data.index.astype(str).str.startswith('PRTC-')
# data = data[keep_mask | prtc_mask].copy()

data.shape


# Filter and normalize the data


In [ ]:
psm_threshold = int(analysis_cfg.get('psm_threshold', 3))
psm_column_contains = analysis_cfg.get('psm_column_contains', 'PSM')
abundance_column_contains = analysis_cfg.get('abundance_column_contains', 'Abundances (Grouped):')
rename_regexes = analysis_cfg.get('column_rename_regexes', [r'Abundances_\(Grouped\):_', r'_Female'])

filtered_psm = filterLowPSM(data, psm_value=psm_threshold, controlColumnContains=psm_column_contains)
abundance_data = extractAbundanceData(filtered_psm, abundanceColumnContains=abundance_column_contains)
renameColumns(abundance_data, regexes=rename_regexes)
abundance_data.head()


# Align data columns with sample metadata


In [ ]:
include_series = sample_metadata['include'].astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y'})
metadata_used = sample_metadata.loc[include_series].copy()

missing_columns = sorted(set(metadata_used['source_column']) - set(abundance_data.columns))
if missing_columns:
    raise KeyError(f'Source columns listed in sample_metadata.csv were not found after column renaming: {missing_columns}')

abundance_data = abundance_data[metadata_used['source_column'].tolist()].copy()
metadata_used['column_name'] = metadata_used['source_column']
metadata_used


# PRTC review and primary normalization


In [ ]:
use_prtc = bool(flags_cfg.get('use_prtc', True))
prtc_mask = abundance_data.index.astype(str).str.startswith('PRTC-')
print('PRTC rows detected:', int(prtc_mask.sum()))

if use_prtc and prtc_mask.any():
    perform_prtc_pca(abundance_data, prtc_mask, n_components=3, perform_log2=True)
    raw_data, normalized_data = normalize_prtc(abundance_data)
else:
    raw_data = abundance_data.copy()
    normalized_data = abundance_data.copy()

plot_data_distribution(raw_data, normalized_data)
normalized_data.head()


# Export QC outputs


In [ ]:
metadata_used.to_csv(qc_dir / 'sample_metadata_used.csv', index=False)
metadata_used.to_csv(qc_dir / 'metadata.txt', index=False)
metadata_used.to_csv(plots_dir / 'metadata.txt', index=False)
annotation.to_csv(qc_dir / 'annotation.csv', index=True)
raw_data.to_csv(qc_dir / 'raw_abundance_matrix.csv', index=True)
normalized_data.to_csv(qc_dir / 'normalized_abundance_matrix.csv', index=True)

with open(qc_dir / 'software_versions.txt', 'w', encoding='utf-8') as handle:
    handle.write(f"python\t{sys.version}\n")
    handle.write(f"numpy\t{np.__version__}\n")
    handle.write(f"pandas\t{pd.__version__}\n")
    handle.write(f"scipy\t{scipy.__version__}\n")
    handle.write(f"statsmodels\t{statsmodels.__version__}\n")
    handle.write(f"plotly\t{plotly.__version__}\n")
    handle.write(f"matplotlib\t{matplotlib.__version__}\n")
    handle.write(f"seaborn\t{sns.__version__}\n")
    handle.write(f"sklearn\t{sklearn.__version__}\n")

print('QC outputs written to', qc_dir)


# Prepare comparison loop


In [ ]:
normalized_no_prtc = normalized_data.loc[~normalized_data.index.astype(str).str.startswith('PRTC-')].copy()
comparison_rows = comparisons[comparisons['enabled'].astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y'})].copy()
comparison_rows


# Run statistics and generate plots


In [ ]:
apply_uq = bool(analysis_cfg.get('apply_uq_per_comparison', True))
astral_mode = bool(flags_cfg.get('astral_mode', False))
default_grouping_column = analysis_cfg.get('grouping_column', 'group')
generate_qvalue_plots = bool(flags_cfg.get('generate_qvalue_plots', True))

comparison_counts = []
output_dirs = {'html': plots_html_dir, 'png': plots_png_dir, 'svg': plots_svg_dir}

for _, row in comparison_rows.iterrows():
    comparison_id = row['comparison_id']
    grouping_column = row.get('grouping_column', default_grouping_column)
    if pd.isna(grouping_column) or str(grouping_column).strip() == '':
        grouping_column = default_grouping_column
    group1 = row['group1']
    group2 = row['group2']
    p_cut = float(row['pvalue_cutoff'])
    q_cut = float(row['qvalue_cutoff'])
    fc_cut = float(row['abs_log2fc_cutoff'])

    print(f'\nProcessing {comparison_id}: {group1} vs {group2}')

    samples_group1 = get_columns_for_group(metadata_used, treatmentGroupColumnID=grouping_column, group_name=group1)
    samples_group2 = get_columns_for_group(metadata_used, treatmentGroupColumnID=grouping_column, group_name=group2)
    if not samples_group1 or not samples_group2:
        raise ValueError(f'Could not resolve samples for comparison {comparison_id}: {group1} vs {group2}')

    combined_samples = samples_group1 + samples_group2
    data_subset = normalized_no_prtc[combined_samples].copy()

    if apply_uq:
        raw_subset, data_subset = uq_normalization(data_subset)
    else:
        raw_subset = data_subset.copy()

    plot_data_distribution(raw_subset, data_subset)

    result_df_t = perform_student_t_test(data_subset, samples_group1, samples_group2, astral=astral_mode)
    result_df_shapiro = test_shapiro_wilk_normality(data_subset, samples_group1, samples_group2)
    result_df_fc = calculate_foldchange(data_subset, samples_group1, samples_group2)

    dfs = [result_df_t, result_df_shapiro, result_df_fc, annotation]
    merged_result = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), dfs)
    merged_result = merged_result.sort_values('p-value_StudentTtest', ascending=True)

    comparison_filename = results_csv_dir / f'{group1}_vs_{group2}_comparison.csv'
    merged_result.to_csv(comparison_filename, index=True)

    p_sig = int(((merged_result['p-value_StudentTtest'] < p_cut) & (merged_result['log2FoldChange'].abs() > fc_cut)).sum())
    q_sig = int(((merged_result['q-value_StudentTtest'] < q_cut) & (merged_result['log2FoldChange'].abs() > fc_cut)).sum())
    comparison_counts.append({
        'comparison_id': comparison_id,
        'grouping_column': grouping_column,
        'group1': group1,
        'group2': group2,
        'pvalue_cutoff': p_cut,
        'qvalue_cutoff': q_cut,
        'abs_log2fc_cutoff': fc_cut,
        'significant_pvalue_hits': p_sig,
        'significant_qvalue_hits': q_sig,
        'output_file': str(comparison_filename.relative_to(project_root)),
    })

    hover_fields = ['Feature', 'Description', 'FoldChange']
    plot_title = f'{group1} vs {group2}'
    plot_stem = f'volcano_plot_{group1}-vs-{group2}'
    plot_volcano(merged_result, columns=['p-value_StudentTtest', 'log2FoldChange'], pvalue_limit=p_cut, fold_change_limit=fc_cut, hoverDataList=hover_fields, plotTitle=plot_title, outFileName=plot_stem, output_dirs=output_dirs)
    if generate_qvalue_plots:
        plot_volcano(merged_result, columns=['q-value_StudentTtest', 'log2FoldChange'], pvalue_limit=q_cut, fold_change_limit=fc_cut, hoverDataList=hover_fields, plotTitle=plot_title, outFileName=plot_stem, q_values=True, output_dirs=output_dirs)

    plot_heatmap(merged_result, columns=['p-value_StudentTtest', 'log2FoldChange'], pvalueLimit=p_cut, foldChangeLimit=fc_cut, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, output_dirs=output_dirs)
    if generate_qvalue_plots:
        plot_heatmap(merged_result, columns=['q-value_StudentTtest', 'log2FoldChange'], pvalueLimit=q_cut, foldChangeLimit=fc_cut, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, q_values=True, output_dirs=output_dirs)

    pc_df, explained_var = compute_pca(data_subset, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, n_components=3, scale_data=True)
    plot_all_2d_pca_pairs(pc_df, comparison=(group1, group2), explained_var=explained_var, output_dirs=output_dirs)
    plot_pca_3d(pc_df, comparison=(group1, group2), explained_var=explained_var, output_dirs=output_dirs)
    plot_pls_da_with_cv(data_subset, comparison=(group1, group2), samples_group1=samples_group1, samples_group2=samples_group2, n_components=2, scale_data=True, n_splits=5, output_dirs=output_dirs)

comparison_counts_df = pd.DataFrame(comparison_counts)
comparison_counts_df


# Organize plot exports and write summary tables


In [ ]:
def move_with_replace(files, destination_dir):
    for file in files:
        target = destination_dir / file.name
        if target.exists():
            target.unlink()
        shutil.move(str(file), target)

move_with_replace(plots_dir.glob('*.png'), plots_png_dir)
move_with_replace(plots_dir.glob('*.svg'), plots_svg_dir)
move_with_replace(plots_dir.glob('*.html'), plots_html_dir)
move_with_replace(plots_dir.glob('*.csv'), plots_csv_dir)

comparison_counts_df.to_csv(results_dir / 'significant_protein_counts.csv', index=False)
comparison_counts_df.to_csv(plots_csv_dir / 'significant_protein_counts.csv', index=False)

print('Results CSV dir:', results_csv_dir)
print('Plots PNG dir:', plots_png_dir)
print('Plots SVG dir:', plots_svg_dir)
print('Plots HTML dir:', plots_html_dir)


# Summary for workflow documentation


In [ ]:
print('## Number of significantly DE proteins')
print(comparison_counts_df[['comparison_id', 'significant_pvalue_hits', 'significant_qvalue_hits']].to_markdown(index=False))

print('\n## Output folders')
print(f'- QC outputs: {qc_dir.relative_to(repo_root)}')
print(f'- Statistical results: {results_dir.relative_to(repo_root)}')
print(f'- Visualization outputs: {plots_dir.relative_to(repo_root)}')
